# __FSO__



## <font color='blue'>__Systematic Topic Review Modeling with MistralAI__</font>




# 📄 **Preamble**

In [ ]:
import pandas as pd

In [ ]:
!pip install -q bertopic bitsandbytes xformers torch==2.5.1

In [ ]:
import transformers
import accelerate
import bitsandbytes
import torch
#import xformers
import huggingface_hub
import openai

In [ ]:
torch.__version__

In [ ]:
transformers.__version__

In [ ]:
#### NO EJECUTAR
%%capture
!pip install bertopic==0.15.0 accelerate==0.23.0 bitsandbytes==0.41.1 xformers==0.0.22 adjustText huggingface_hub==0.19.4 openai

In [ ]:
#### NO EJECUTAR
%%capture
!pip install transformers==4.36.1

Keep in mind that transformers, by design, include several stochastic components such as weight initialization, dropout layers, and the sequencing and batching of training. This could result in different outputs each time the model is run.



## **Data**



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
file_path = '/content/drive/MyDrive/_RESEACH/_FSO/data/df_combined2.pkl'
file_path_2 = '/content/drive/MyDrive/_RESEACH/_FSO/data/df.xlsx'

In [ ]:
df = pd.read_excel(file_path_2)
df['file_number'] = df['file_number'].astype(int).astype(str)

df_combined = pd.DataFrame()
df_combined['Doc_Id'] = df['file_number']
df_combined['Title'] = df['Article Title']
df_combined['Authors'] = df['Authors']
df_combined['Year'] = df['Publication Year']
df_combined['DOI'] = df['DOI']
df_combined['Abstract'] = df['Abstract_y']
df_combined['Citations'] = df['Times Cited, All Databases']
df_combined['Keywords'] = df['Proposed Keywords']
# Create a new column 'Full_Text' by concatenating selected columns
df_combined['Full_Text'] = df[['Article Title', 'Abstract_y', 'Main Research Question',
                                        'Methods and Experiments', 'Results',
                                        'Discussion', 'Conclusions', 'Future Work']].apply(lambda x: ' '.join(x), axis=1)

In [ ]:
df.head(5)

In [ ]:
df_combined.info()

In [ ]:
df_combined.to_pickle('/content/drive/MyDrive/_RESEACH/_FSO/data/df_combined.pkl')


In [ ]:
abstracts = df_combined["Full_Text"].to_list()
titles = df_combined["Title"].to_list()

Titulo y Abstract, ejemplo

In [ ]:
print(titles[1])
print(abstracts[1])

## Login HuggingFace

In [ ]:
#hugging face token
token_hf="<Your API Key>"
from huggingface_hub import login
login(token_hf) #, add_to_git_credential=True)

**Mistral**

In [ ]:
from torch import cuda
device = f'cuda:{cuda.current_device()}' if cuda.is_available() else 'cpu'

print(device)

## **Optimization & Quantization**

To load our model, we'll need to implement some optimization tricks. Given that we have limited VRAM and not an A100 GPU, we will need to "condense" the model a bit so that we can run it.

There are several tricks we can use, but the main one will be 4-bit quantization.

This process reduces the 64-bit representation to just 4 bits, which decreases the GPU memory we will need. It's a recent and quite elegant technique for efficiently loading and using massive language models. You can learn more about this method [here](https://arxiv.org/pdf/2305.14314.pdf) in the QLoRA paper and on the amazing HuggingFace blog [here](https://huggingface.co/blog/4bit-transformers-bitsandbytes).

In [ ]:
from torch import bfloat16
import transformers

# set quantization configuration to load large model with less GPU memory
# this requires the `bitsandbytes` library

bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,  # 4-bit quantization
    bnb_4bit_quant_type='nf4',  # Normalized float 4
    bnb_4bit_use_double_quant=True,  # Second quantization after the first
    bnb_4bit_compute_dtype=bfloat16  # Computation type
)

These four parameters we just executed are incredibly important and offer many applications of massive language models (LLM) to consumers:

* `load_in_4bit`
  * This allows us to load the model with 4-bit precision compared to the original 32-bit precision.
  * This provides us with an incredible speed boost and reduces the required memory.

* `bnb_4bit_quant_type`
  * This is the type of 4-bit precision. The article recommends using normalized floating 4-bit, so that's what we're going to use.

* `bnb_4bit_use_double_quant`
  * This is an interesting trick as it performs a second quantization after the first, further reducing the amount of bits needed.

* `bnb_4bit_compute_dtype`
  * The compute type used during computation, which further speeds up the model.



Using this configuration, we can start loading the model and the tokenizer:

In [ ]:
import torch
if torch.cuda.is_available():

    print(f"Número de GPUs disponibles: {torch.cuda.device_count()}")
    print(f"Nombre de la GPU actual: {torch.cuda.get_device_name(0)}")
    print(f"Capacidad de memoria de la GPU actual: {torch.cuda.get_device_properties(0).total_memory / (1024 ** 2):.2f} MB")
else:
    print("No se encontraron GPUs disponibles en este sistema.")


In [ ]:
from huggingface_hub import HfApi

def check_model_availability(model_id):
    api = HfApi()
    try:
        model_info = api.model_info(model_id)
        return f"Model '{model_id}' is available on Hugging Face."
    except Exception as e:
        return f"Model '{model_id}' is not available on Hugging Face. Error: {e}"

model_id = "upstage/SOLAR-10.7B-Instruct-v1.0"
model_id = 'mistralai/Mistral-7B-Instruct-v0.1'
model_id = 'mistralai/Mistral-7B-Instruct-v0.3'
#model_id = 'mistralai/Mistral-Small-24B-Instruct-2501'
#model_id = 'Qwen/Qwen2.5-32B-Instruct'
print(check_model_availability(model_id))


In [ ]:
# Llama 2 Tokenizer
from torch import bfloat16
import transformers


tokenizer = transformers.AutoTokenizer.from_pretrained(model_id)

# Mistral Model
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
#    quantization_config=bnb_config,
    device_map=None #'auto'
)
model.eval()

In [ ]:
print(model)

Using the model and tokenizer, we will create a HuggingFace transformers pipeline that will allow us to easily generate new text:

Here are some of the specific parameters being used in the provided code:

- **task='text-generation'**: This setting indicates that the model will be used for the task of text generation. In other words, the model is expected to generate text by continuing from a given input.
- **do_sample=True**: This parameter indicates that random sampling will be used to generate text. Instead of always generating the most likely output, the model will randomly select from the available options, which can lead to more diversified results.
- **temperature=0.1**: Temperature is a parameter that controls the randomness in the generation process. A low value like 0.1 makes the generation more deterministic and tends to produce more coherent results.
- **max_new_tokens=500**: This limits the maximum number of tokens (words or sub-units) that will be generated in the output. In this case, the generation is limited to a maximum of 500 tokens.
- **repetition_penalty=1.1**: Repetition penalty is a value used to penalize excessive repetition of certain tokens in generation. A value greater than 1.0, such as 1.1, slightly penalizes repetitions in the generation.

In [ ]:
# Our text generator
generator = transformers.pipeline(
    model=model,
    tokenizer=tokenizer,
    task='text-generation',
    #do_sample=True,
    temperature=0.1,
    max_new_tokens=500,
    repetition_penalty=1.1,
)

In [ ]:
model.generation_config

## **Prompt Engineering**

Although we can give direct instructions to the model, there's actually a template we need to follow. The template looks like this:

```python
"""
<s>[INST] <<SYS>>

{{ System Prompt }}

<</SYS>>

{{ User Prompt }} [/INST]

{{ Model Answer }}
"""
```

EThis template consists of two main components, namely the `{{ System Prompt }}` and the `{{ User Prompt }}`:
* The `{{ System Prompt }}` helps us guide the model during a conversation. For example, we can state that it is a helpful assistant specialized in labeling topics.
* The `{{ User Prompt }}` is where we ask a question.

You may have noticed the `[INST]` tags, which are used to identify the start and end of an instruction. We can use them to model the conversation history, as we will see in more detail later.

Next, let's see how we can use this template to optimize Mistral for topic modeling.

### **Prompt Template**


In [ ]:
# System prompt describes information given to all conversations
system_prompt = """
<s>[INST] <<SYS>>
You are a helpful, respectful and honest assistant for labeling topics.
<</SYS>>
"""

# Example prompt demonstrating the output we are looking for
example_prompt = """
I have a topic that contains the following documents:
- Free-space optical (FSO) communication is a technology that uses light propagation through the atmosphere or space to transmit data wirelessly.
- Atmospheric conditions such as fog, rain, and turbulence can degrade FSO link performance and impact reliability.
- FSO technology is widely used in satellite communication, urban networking, and secure military applications due to its high data rate and low electromagnetic interference.

The topic is described by the following keywords: 'FSO, free-space optics, communication, laser, atmospheric, satellite, wireless, data, transmission, security'.

Based on the information about the topic above, please create a short label of this topic. Make sure you to only return the label and nothing more.

[/INST] Challenges and applications of FSO communication
"""

# Our main prompt with documents ([DOCUMENTS]) and keywords ([KEYWORDS]) tags
main_prompt = """
[INST]
I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: '[KEYWORDS]'.

Based on the information about the topic above, please create a short label of this topic. Make sure you to only return the label and nothing more.
[/INST]
"""

prompt = system_prompt + example_prompt + main_prompt

There are two specific BERTopic tags of interest, namely `[DOCUMENTS]` and `[KEYWORDS]`:

* `[DOCUMENTS]` contains the 5 most relevant documents for the topic.
* `[KEYWORDS]` contains the 10 most relevant keywords for the topic generated through c-TF-IDF.


# 🗨️ **BERTopic**

Before we begin with topic modeling, we must first perform two steps:
* Precompute embeddings.
* Define submodels.

By precomputing embeddings for each document, we can accelerate further exploration steps and use the embeddings to quickly iterate over BERTopic's hyperparameters if necessary.

In [ ]:
!pip install -U cohere

In [ ]:
import cohere
import numpy as np
import time

In [ ]:
cohere_key = "<Your API Key>"
co = cohere.Client(cohere_key)



In [ ]:
def chunk_list(lst, step_size):
    """Divide la lista en segmentos superpuestos avanzando de 'step_size' en 'step_size'."""
    for i in range(0, len(lst), step_size):
        yield lst[i:i + step_size]

# Generar embeddings en lotes para evitar el límite
embeddings = []
step_size = 5  # Avanzar de 5 en 5 abstracts

for batch in chunk_list(abstracts, step_size):
    response = co.embed(
        texts=batch,
        model="embed-english-v3.0",
        input_type="search_document"
    )
    embeddings.extend(response.embeddings)
    time.sleep(5)  # Espera para evitar límite de rate

embeddings = np.array(embeddings)

In [ ]:
print(embeddings.shape)  #


In [ ]:
### DELETE #####
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic

# Definir el modelo de embeddings (ejemplo con Cohere o BGE)
embedding_model = SentenceTransformer("Cohere/embed-english-v3.0")  # Máximo 4096 tokens

# Lista de abstracts (cada uno se procesa por separado)


# Generar embeddings (cada texto se trunca a 4096 tokens si es más largo)
embeddings = embedding_model.encode(abstracts)



In [ ]:
### DELETE #####
from sentence_transformers import SentenceTransformer

# Pre-calculate embeddings
embedding_model = SentenceTransformer("BAAI/bge-large-en-v1.5") # Postion 14 in Leaderboar. Embedding dimension = 1024, max imput tokens = 512
# The model is transformed into an embedding using the previous transformer.
embeddings = embedding_model.encode(abstracts, show_progress_bar=True)

## **Sub-models**

Next, we will define all the submodels in BERTopic and make some minor adjustments to the number of clusters to be created, setting random_state, etc.

__UMAP Parameters__

- __n_neighbors__: This parameter controls the size of the local neighborhood from which each point draws information to be projected into a low-dimensional space. A higher value considers a broader context, making the global structure more prominent in the low-dimensional representation. A lower value causes the projection to focus more on the local structure, potentially revealing clusters or local patterns at the expense of losing the overall view of the global structure.

- __n_components__: Specifies the number of dimensions of the projected space (i.e., the low-dimensional space to which the data will be reduced). For example, n_components=2 will project the data onto a two-dimensional plane, which is useful for visualizations. Increasing n_components allows capturing more details of the data structure in more dimensions, but it can make visualizing and interpreting the results more challenging.



In [ ]:
from umap import UMAP
from hdbscan import HDBSCAN

# Dimensionality reduction
umap_model = UMAP(n_neighbors=5, n_components=6, min_dist=0.0, metric='cosine', random_state=42) # before n_neighbors = 10
# Clustering - Hierarchical Density-Based Spatial Clustering of Applications with Noise
hdbscan_model = HDBSCAN(min_cluster_size=3, metric='euclidean', cluster_selection_method='eom', prediction_data=True) # before min_cluster_size=120

As a small additional benefit, we will reduce the embeddings we created earlier to 2 dimensions so that we can use them for visualization purposes once we have created our topics.

In [ ]:
# Pre-reduce embeddings for visualization purposes
reduced_embeddings = UMAP(n_neighbors=5, n_components=6, min_dist=0.0, metric='cosine').fit_transform(embeddings) # before n_neighbors = 15 or 4

### **Representation Models**

One of the ways we are going to represent the topics is with SOLAR, which should provide us with a clear label. However, we may want to have additional representations to view a topic from multiple perspectives.

Here, we are going to use c-TF-IDF as our main representation and [KeyBERT](https://maartengr.github.io/BERTopic/getting_started/representation/representation.html#keybertinspired), [MMR](https://maartengr.github.io/BERTopic/getting_started/representation/representation.html#maximalmarginalrelevance), and [LLM](https://maartengr.github.io/BERTopic/getting_started/representation/llm.html) as our additional representations.

Three different representation models are used because each provides a unique perspective and can be useful for different tasks and objectives in topic modeling. Here is a brief explanation of why three models are used:

1. **KeyBERTInspired (KeyBERT)**: This model is based on the idea of extracting keywords from documents. It provides a keyword-based representation that highlights the most important words in a document.

2. **MaximalMarginalRelevance (MMR)**: MMR focuses on maximizing the relevance of selected documents while maintaining adequate diversity. It allows selecting documents that are relevant but also diverse, which can be useful to avoid redundancy in the results of the topic clustering.

3. **Text Generation with Llama 2 (Llama2)**: Llama 2 is a text generator that can be used to generate longer text representations or to expand topic descriptions. It provides a more generative perspective that can be useful for creating summaries or expanding topics in detail.

In [ ]:
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, TextGeneration

# KeyBERT
#keybert = KeyBERTInspired()

# MMR
#mmr = MaximalMarginalRelevance(diversity=0.3)

# Text generation with Mistral
llmodel = TextGeneration(generator, prompt=prompt)

# All representation models
representation_model = {
#    "KeyBERT": keybert,
    "Mistral": llmodel,
#    "MMR": mmr,
}

## **Training**

Now that we have our models prepared, we can start training our topic model! We supply BERTopic with the submodels of interest, run `.fit_transform`, and observe what kinds of topics we get.

In [ ]:
from bertopic import BERTopic

topic_model = BERTopic(

  # Sub-models
  embedding_model=None, #embedding_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  representation_model=representation_model,

  # Hyperparameters
  top_n_words=10,
  verbose=True
)

# Train model
topics, probs = topic_model.fit_transform(abstracts, embeddings=embeddings)


Now that we have finished training our model, let's see what topics were generated:


In [ ]:
len(abstracts)

In [ ]:
topic_model.get_topic_info()

In [ ]:
# Show topics
topic_model.get_topic_info()

In [ ]:
topic_info = topic_model.get_topic_info()
topic_info["Mistral"] = topic_info["Mistral"].apply(lambda x: x[0].lstrip('\n') if isinstance(x, list) and x else "")


In [ ]:
topic_info.to_excel('/content/drive/MyDrive/_RESEACH/_FSO/topic_info.xlsx')


In [ ]:
topic_model.get_topic_info().iloc[3]['Mistral']

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.get_topic(1, full=True)["KeyBERT"] # Old run

We got over 100 topics that were created and they all seem quite diverse.We can use the labels by Llama 2 and assign them to topics that we have created. Normally, the default topic representation would be c-TF-IDF, but we will focus on Llama 2 representations instead.


In [ ]:
labels = [label[0][0].lstrip("\n") for label in topic_model.get_topics(full=True)["Mistral"].values()]
topic_model.set_topic_labels(labels)

In [ ]:
labels # New

## **Visualize**
We can go through each topic manually, which would take a lot of work, or we can visualize them all in a single interactive graph.
BERTopic has a bunch of [visualization functions](https://medium.com/r/?url=https%3A%2F%2Fmaartengr.github.io%2FBERTopic%2Fgetting_started%2Fvisualization%2Fvisualize_documents.html) that we can use. For now, we are sticking with visualizing the documents.

In [ ]:
# topic_model.visualize_documents(abstracts, reduced_embeddings=reduced_embeddings, hide_annotations=True, hide_document_hover=False, custom_labels=True)

In [ ]:
topic_model.visualize_documents(titles, reduced_embeddings=reduced_embeddings, hide_annotations=True, hide_document_hover=False, custom_labels=True)

In [ ]:
# prompt: topic_model.visualize_documents to html

# Assuming you have already executed the provided code and have the topic_model and reduced_embeddings variables defined.

from IPython.display import HTML

html_visualization = topic_model.visualize_documents(titles, reduced_embeddings=reduced_embeddings, hide_annotations=True, hide_document_hover=False, custom_labels=True)

HTML(html_visualization.to_html())


In [ ]:
# Guardar la visualización en un archivo HTML
file_path = "/content/drive/MyDrive/_RESEACH/_FSO/topic_visualization.html"
with open(file_path, "w", encoding="utf-8") as f:
    f.write(html_visualization.to_html())

## **Advanced Visualization**

Although we can use the built-in visualization features of BERTopic, we can also create a static visualization that might be a bit more informative.

We start by creating the necessary variables that contain our reduced embeddings and representations:

In [ ]:
topics_documents = topic_model.get_document_info(abstracts)

In [ ]:
hierarchical_topics = topic_model.hierarchical_topics(abstracts)
vis2 = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics, custom_labels=True)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics, custom_labels=True)

Each level represents a grouping of documents and topics at a specific level of detail or abstraction. As you move from level 0 to level 9, you are exploring the topics at different levels of granularity.



topic_model.set_topic_labels({1: "Space Travel", 7: "Religion"})

topic_model.visualize_topics_over_time(topics_over_time, topics=[9, 10, 72, 83, 87, 91])

In [ ]:
similar_topics, similarity = topic_model.find_topics("Transformers", top_n=5)
top_topics = []
for topic in similar_topics:
  top_topics.append(topic_model.get_topic(topic))
top_topics


### __Dump files__

In [ ]:
topics_documents

In [ ]:
import pickle

with open('/content/drive/MyDrive/_RESEACH/_FSO/topics_documents_0218.pkl', 'wb') as f:
    pickle.dump(topics_documents, f)


In [ ]:
info_df = topic_model.get_topic_info()

In [ ]:
info_df

In [ ]:
total_count = info_df['Count'].sum()

info_df['percentage'] = (info_df['Count'] / total_count)
info_df

In [ ]:
with open('/content/drive/MyDrive/_RESEACH/_FSO/info_df_0218.pkl', 'wb') as f:
    pickle.dump(info_df, f)


In [ ]:
info_df.to_csv('info_model_bertopic_solar.csv', index=False)
topics_documents.to_csv('document_topics_abstracts.csv', index=False)
df.to_csv(path_file + 'espacio.csv', index=False)


## Visualization over time

In [ ]:
df_combined.info()

In [ ]:
timestamps = pd.to_datetime(df_combined.Year, format='%Y', errors='coerce').to_list()
abs = df_combined.Abstract.to_list()

# Create topics over time
model2 = BERTopic(verbose=True)
topics, probs = model2.fit_transform(abs)
topics_over_time = model2.topics_over_time(abs, timestamps)

In [ ]:
print(topics_over_time["Topic"].value_counts())  # Ver cuántos tópicos aparecen

In [ ]:
import plotly.io as pio

vis4 = model2.visualize_topics_over_time(topics_over_time, topics=range(15), custom_labels=True)
pio.write_html(vis4, "topics_over_time.html")
model2.visualize_topics_over_time(topics_over_time, topics=range(15), custom_labels=True)

In [ ]:
df_subtopic = topics_documents

In [ ]:
merge_df = pd.concat([dataset, df_subtopic], axis=1)

In [ ]:
merge_df.columns

## Relevance Index

The metric that sorts papers by order of relevance:
\[
\text{Relevance Index} = \frac{\text{Cited by}}{\text{Years since publication}}
\]

- **Cited by:** This component measures the influence and recognition a paper has received from the academic community. The more citations it has, the higher this value, indicating greater visibility and perceived relevance by other researchers.

- **Years since publication:** This component represents the time that has elapsed since the paper was published. The idea is that over time, the relevance of a paper can decrease as new advancements and discoveries may have been made.

- **Relevance Index:** The metric itself is a ratio between the citations received and the time elapsed since publication. A higher Relevance Index indicates that the paper has maintained or even increased its significance over time compared to other papers.

**Examples:**

1. **Paper A:** Published 5 years ago, has received 100 citations.
$$
    \text{Relevance Index}_A = \frac{100}{5} = 20
$$

2. **Paper B:** Published 2 years ago, has received 50 citations.
$$
    \text{Relevance Index}_B = \frac{50}{2} = 25
$$


In [ ]:
import datetime

current_year = datetime.datetime.now().year

# Calcular la diferencia entre el año actual y el año de publicación
merge_df['Years_since_publication'] = current_year - merge_df['Year']

# Calcular el índice de relevancia
merge_df['Relevance_Index'] = merge_df['Cited by'] / merge_df['Years_since_publication'].replace(0, 1)


In [ ]:
merge_df[['Title', 'Year', 'Cited by', 'Years_since_publication', 'Relevance_Index']]

In [ ]:
slr_df = merge_df[merge_df["Topic"]==14].sort_values(by="Relevance_Index", ascending = False)
slr_df = slr_df[['Title', 'Year', 'Cited by', 'Abstract', 'Author Keywords','Index Keywords' , 'Document Type', 'Probability', 'Representative_document', 'Relevance_Index']]

In [ ]:
slr_df.to_excel("subtopic14_SLR.xlsx")